In [1]:
# ==========================================================
# Imports
# ==========================================================

import json
import pickle

import numpy as np
import pandas as pd

from xgboost import XGBClassifier

In [2]:
# ==========================================================
# Load Dataset
# ==========================================================

MASTER = "/kaggle/input/notebooks/shri7ul/13-semantic-features-ipynb/master_semantic.parquet"
# MASTER = "/kaggle/input/datasets/shri7ul/master-semantic-parque"


master = pd.read_parquet(MASTER)

print(master.shape)
master.head()

(35072, 154)


,response_id,session_id,learning_objective_id,learning_objective,transcript,student_text,tutor_text,background_text,is_correct,objective_frequency,...,obj_emb_22,obj_emb_23,obj_emb_24,obj_emb_25,obj_emb_26,obj_emb_27,obj_emb_28,obj_emb_29,obj_emb_30,obj_emb_31
0,aaaavsh,bcaufvc,dqibnvd,Knowing the value of each digit in numbers wit...,"[BACKGROUND] [unclear]\n[TUTOR] Miss, I can't ...","Can you hear me? Hello? Yeah, I hear you. Can ...","Miss, I can't hear. Can you hear me? [unclear]...","[unclear] Good, very good. [unclear] And we wi...",1.0,1267,...,-0.036747,0.050525,-0.142228,-0.034155,-0.041571,0.096069,-0.106509,-0.002536,0.018404,-0.097689
1,aaabhzi,eyutanf,eukmzxl,Adding and subtracting tens to a 2-digit number.,[TUTOR] Yay! Hello!\n[STUDENT] Hello.\n[TUTOR]...,Hello. Hello. I was one minute early. I'm one ...,Yay! Hello! Hello. Hello. [Speaker:Background]...,20. What 2-digit number can you make using the...,1.0,44,...,-0.084421,0.075249,-0.007155,-0.005594,-0.062537,0.049526,0.039313,-0.047113,0.009421,-0.103666
2,aaahpnz,juptkxd,fjbqcsv,Comparing and ordering fractions by finding a ...,[BACKGROUND] [unclear]\n[TUTOR] Is it me you a...,Hello. Hello. Yes. Good. How are you? Normal. ...,"Is it me you are looking for? Okay, so did you...","[unclear] Hello. Hello, Callum. Can you hear m...",0.0,211,...,0.062304,0.119096,0.062892,-0.044874,0.084868,-0.006301,-0.056318,0.054294,0.066929,0.005327
3,aaajpom,ntwkcfj,acvbcev,Comparing fractions using reasoning.,[BACKGROUND] [unclear]\n[TUTOR] Hello?\n[STUDE...,"Hello, Tobias. Can you hear me? Okay, that's n...","Hello? Yes, I can hear you. Good. Why are you ...","[unclear] 1.45 meters. Excellent. Yes, good. G...",0.0,510,...,0.073598,0.026815,-0.085554,0.098834,0.027007,-0.040452,-0.025349,0.007395,-0.001951,-0.020720
4,aaamwux,jqriibm,krfuudx,Counting in multiples.,[BACKGROUND] [unclear]\n[TUTOR] Hello.\n[STUDE...,"Hello. Good. Okay, how was this week for you? ...",Hello. Hello. Are you feeling tired today? Is ...,"[unclear] Hello. Hi, Caelum. How are you doing...",0.0,525,...,-0.030643,-0.070860,-0.079507,-0.059127,0.189734,-0.102675,0.121264,0.161541,-0.122001,-0.095332


In [3]:
# ==========================================================
# Prepare Features
# ==========================================================

TARGET = "is_correct"

DROP_COLUMNS = [

    "response_id",
    "session_id",

    "learning_objective",
    "learning_objective_id",

    "transcript",
    "student_text",
    "tutor_text",
    "background_text",

    TARGET,
]

X = master.drop(columns=DROP_COLUMNS)

y = master[TARGET].astype(int)

print(X.shape)
print(y.shape)

(35072, 145)
(35072,)


In [4]:
# ==========================================================
# Encode Categories
# ==========================================================

category_mapping = {}

cat_cols = X.select_dtypes(include="object").columns

for col in cat_cols:

    X[col] = X[col].astype("category")

    category_mapping[col] = (
        X[col]
        .cat
        .categories
        .tolist()
    )

    X[col] = X[col].cat.codes

print(cat_cols.tolist())

['objective_family']


In [5]:
# ==========================================================
# Best Parameters
# ==========================================================

BEST_PARAMS = {

    "n_estimators":5000,

    "learning_rate":0.02,

    "max_depth":7,

    "min_child_weight":1,

    "subsample":0.8,

    "colsample_bytree":0.6,

    "objective":"binary:logistic",

    "eval_metric":"logloss",

    "tree_method":"hist",

    "random_state":42,
}

In [6]:
# ==========================================================
# Train Final Model
# ==========================================================

model = XGBClassifier(**BEST_PARAMS)

model.fit(

    X,
    y,

    verbose=False,

)

print("Training Complete ✓")

Training Complete ✓


In [7]:
# ==========================================================
# Save Assets
# ==========================================================

model.save_model(

    "/kaggle/working/xgb_model.json"

)

with open(

    "/kaggle/working/feature_columns.pkl",

    "wb",

) as f:

    pickle.dump(

        X.columns.tolist(),

        f,

    )

with open(

    "/kaggle/working/category_mapping.pkl",

    "wb",

) as f:

    pickle.dump(

        category_mapping,

        f,

    )

with open(

    "/kaggle/working/best_params.json",

    "w",

) as f:

    json.dump(

        BEST_PARAMS,

        f,

        indent=4,

    )

print("Assets Saved ✓")

Assets Saved ✓


In [8]:
# ==========================================================
# Verify
# ==========================================================

import os

for file in sorted(os.listdir("/kaggle/working")):

    print(file)

__notebook__.ipynb
best_params.json
category_mapping.pkl
feature_columns.pkl
xgb_model.json
